# Pinecone RAG Testing — Clinical Trial Document Retrieval

This notebook tests Pinecone retrieval for your ctdgen project.  
It matches the exact embedding model (`all-MiniLM-L6-v2`, 384-dim) your FAISS pipeline already uses.  
Output is structured as section dicts ready to drop into your existing `GenerationResponse` schema.

**Workflow:**
1. Connect to Pinecone
2. Embed a study query using the same encoder as your backend
3. Retrieve matching historical trial chunks
4. Group chunks into section-shaped output
5. Test with multiple indication queries
6. Integration snippet for `rag_service.py`

## 0. Install dependencies

In [1]:
# Run once — skip if already installed
!pip install pinecone-client sentence-transformers pandas tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\BasanthKumarPutta\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


## 1. Config — fill your Pinecone credentials here

In [13]:
# ─── FILL THESE IN ────────────────────────────────────────────────────────────
PINECONE_API_KEY   = "pcsk_5K2TpT_AMzJu3syYmv3k9W5HkqatuyaaTcScNyd9sxvzkM4WUsW8JjB9NYZ8TSopuSeDZN"
PINECONE_INDEX     = "clinical-rag-index"          # e.g. 'ctdgen-trials'
PINECONE_NAMESPACE = "Chenodeoxycholic_chunks"                          # leave empty string if not using namespaces
EMBED_MODEL        = "BAAI/bge-base-en-v1.5"          # must match your FAISS backend
VECTOR_DIM         = 768                         # must match your index dimension
TOP_K              = 10                          # chunks to retrieve per query
# ──────────────────────────────────────────────────────────────────────────────

print(f"Config loaded — index: {PINECONE_INDEX}, model: {EMBED_MODEL}, dim: {VECTOR_DIM}")

Config loaded — index: clinical-rag-index, model: BAAI/bge-base-en-v1.5, dim: 768


## 2. Connect to Pinecone

In [6]:
from pinecone import Pinecone

pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

# Sanity-check: print index stats
stats = index.describe_index_stats()
print("✅ Connected to Pinecone")
print(f"   Total vectors : {stats['total_vector_count']:,}")
print(f"   Dimension     : {stats['dimension']}")
print(f"   Namespaces    : {list(stats.get('namespaces', {}).keys()) or ['(default)']}")

assert stats['dimension'] == VECTOR_DIM, \
    f"Dimension mismatch! Index={stats['dimension']} but VECTOR_DIM={VECTOR_DIM}"

✅ Connected to Pinecone
   Total vectors : 1,091
   Dimension     : 768
   Namespaces    : ['Amlodipine_sections', 'Epclusa_v1_sections', 'Chenodeoxycholic_chunks', 'Bimervax_sections', 'Amlodipine_chunks', 'Epclusa_v1_chunks', 'Bimervax_chunks', 'Chenodeoxycholic_sections']


## 3. Load the embedding model
Same `all-MiniLM-L6-v2` your `rag_service.py` uses — so query embeddings are always compatible.

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer(EMBED_MODEL)
print(f"✅ Encoder loaded: {EMBED_MODEL}")

def embed(text: str) -> list:
    """Embed a single string, L2-normalised (matches FAISS IndexFlatIP behaviour)."""
    vec = encoder.encode([text], normalize_embeddings=True)
    return vec[0].tolist()

c:\Users\BasanthKumarPutta\Desktop\AI medical writing\ctdgen-full\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BasanthKumarPutta\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package 

✅ Encoder loaded: BAAI/bge-base-en-v1.5


## 4. Query builder — mirrors `StudyMetadata.to_query_text()` from your backend

In [8]:
def build_query(metadata: dict) -> str:
    """
    Replicates StudyMetadata.to_query_text() from backend/core/models.py.
    Pass a dict with the same keys as StudyMetadata.
    """
    parts = [
        f"Indication: {metadata.get('indication', '')}",
        f"Phase: {metadata.get('phase', '')}",
        f"Design: {metadata.get('design', '')}",
        f"Primary Endpoint: {metadata.get('primary_endpoint', '')}",
        f"Population: {metadata.get('patient_population', '')}",
    ]
    if metadata.get('secondary_endpoints'):
        eps = metadata['secondary_endpoints']
        if isinstance(eps, list):
            eps = ', '.join(eps)
        parts.append(f"Secondary Endpoints: {eps}")
    if metadata.get('investigational_product'):
        parts.append(f"Drug: {metadata['investigational_product']}")
    if metadata.get('therapeutic_area'):
        parts.append(f"Therapeutic Area: {metadata['therapeutic_area']}")
    return " | ".join(parts)


# ── Sample query metadata — edit these for your test ─────────────────────────
SAMPLE_METADATA = {
    "indication":              "Cerebrotendinous Xanthomatosis (CTX)",
    "therapeutic_area":        "Rare Metabolic Disorders / Neurology",
    "phase":                   "Phase III",
    "design":                  "Randomized, Double-blind, Placebo-controlled, Multicenter",
    "primary_endpoint":        "Reduction in plasma cholestanol levels",
    "secondary_endpoints": [
        "Neurological function improvement (disability scales)",
        "MRI-based disease progression",
        "Bone mineral density changes",
        "Safety and tolerability (AE/SAE incidence)"
    ],
    "patient_population":      "Patients aged 2–75 years diagnosed with CTX, confirmed via biochemical/genetic testing",
    "investigational_product": "Chenodeoxycholic Acid (CDCA)",
    "sample_size":             120,
    "duration_months":         36,
    "treatment_arms": [
        "CDCA 750 mg/day orally",
        "Placebo"
    ],
    "sponsor":                 "Sigma-Tau Research Switzerland S.A."
}

query_text = build_query(SAMPLE_METADATA)
print("Query text:")
print(query_text)

Query text:
Indication: Cerebrotendinous Xanthomatosis (CTX) | Phase: Phase III | Design: Randomized, Double-blind, Placebo-controlled, Multicenter | Primary Endpoint: Reduction in plasma cholestanol levels | Population: Patients aged 2–75 years diagnosed with CTX, confirmed via biochemical/genetic testing | Secondary Endpoints: Neurological function improvement (disability scales), MRI-based disease progression, Bone mineral density changes, Safety and tolerability (AE/SAE incidence) | Drug: Chenodeoxycholic Acid (CDCA) | Therapeutic Area: Rare Metabolic Disorders / Neurology


## 5. Core retrieval function

In [14]:
def retrieve_from_pinecone(
    query: str,
    top_k: int = TOP_K,
    metadata_filter: dict = None,
    namespace: str = PINECONE_NAMESPACE,
) -> list:
    """
    Embed query and retrieve matching chunks from Pinecone.

    Returns:
        List of dicts:
            {
              'id':       str,          # Pinecone vector ID
              'score':    float,        # cosine similarity (higher = more relevant)
              'text':     str,          # chunk content (from metadata['text'])
              'metadata': dict,         # all metadata stored with the vector
            }
    """
    vec = embed(query)

    kwargs = dict(
        vector          = vec,
        top_k           = top_k,
        include_metadata= True,
    )
    if namespace:
        kwargs['namespace'] = namespace
    if metadata_filter:
        kwargs['filter'] = metadata_filter

    response = index.query(**kwargs)

    results = []
    for match in response.get('matches', []):
        meta = match.get('metadata', {})
        results.append({
            'id':       match['id'],
            'score':    round(match['score'], 4),
            'text':     meta.get('text', meta.get('content', '')),  # support both key names
            'metadata': meta,
        })
    return results


print("retrieve_from_pinecone() defined ✅")

retrieve_from_pinecone() defined ✅


## 6. Run the sample query and inspect raw results

In [15]:
import pandas as pd

results = retrieve_from_pinecone(query_text)

print(f"\n✅ Retrieved {len(results)} chunks\n")

# Display as a table for easy inspection
rows = []
for r in results:
    rows.append({
        'id':         r['id'],
        'score':      r['score'],
        'indication': r['metadata'].get('indication', '—'),
        'phase':      r['metadata'].get('phase', '—'),
        'doc_type':   r['metadata'].get('doc_type', '—'),
        'section':    r['metadata'].get('section_type', r['metadata'].get('section', '—')),
        'source':     r['metadata'].get('filename', r['metadata'].get('source', '—')),
        'text_preview': r['text'][:120].replace('\n', ' ') + '…' if len(r['text']) > 120 else r['text'],
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)
display(df)


✅ Retrieved 10 chunks



,id,score,indication,phase,doc_type,section,source,text_preview
0,3_text_0,0.8365,—,—,—,—,—,This is a retrospective single arm cohort study collecti...
1,16_text_3,0.8335,—,—,—,—,—,(CDCA) in Patients Affected by CerebrotendinousXanthomat...
2,16_text_11,0.8328,—,—,—,—,—,"Velden MGVerripsAPrinsen BH, de Barse M,Berger R,Visser ..."
3,1_text_0,0.8244,—,—,—,—,—,1.1BasicInformationaboutCerebrotendinous Xanthomatosisan...
4,16_text_5,0.8142,—,—,—,—,—,cholestanol in cerebrotendinous xanthomatosis.J Clin Inv...
5,1.1.4_text_7,0.8123,—,—,—,—,—,anP possibly put the patient at risk for treatment compl...
6,16_text_14,0.8099,—,—,—,—,—,J Neuroradiol.Mar 2003;243495-500.[Medline] 58. Pilo Bde...
7,1.1.4_text_8,0.8090,—,—,—,—,—,recent study demonstratepatients who started treatment l...
8,16_text_12,0.8069,—,—,—,—,—,"Skrede S,Sjovall J.Fast atom bombardment mass spectromet..."
9,16_text_2,0.8054,—,—,—,—,—,ProtocolCDCA-STRCH-CR-14-001 GENERAL INFORMATION reof Pr...


In [19]:
results

[{'id': '3_text_0',
  'score': 0.8365,
  'text': 'This is a retrospective single arm cohort study collecting clinical, laboratory and physiological data generated in a group of patient affected by cerebrotendinous xanthomatosis, treated with CDCA orally at 750 mg/day followed in the Operative Unit of Neurology and Neuronretabolic Diseases, Medical University of Siena, Policlinico Le ScotteSiena Italy) any anda',
  'metadata': {'chunk_id': '3_text_0',
   'chunk_type': 'text',
   'depth': 1,
   'document_id': 'Chenodeoxycholic',
   'page_ids': '2,19',
   'parent_id': '',
   'row_id': '',
   'section_id': '3',
   'section_path': '',
   'section_title': 'EXPERIMENTAL DESIGN',
   'source_file': 'data/documents/Chenodeoxycholic.pdf',
   'table_id': '',
   'text': 'This is a retrospective single arm cohort study collecting clinical, laboratory and physiological data generated in a group of patient affected by cerebrotendinous xanthomatosis, treated with CDCA orally at 750 mg/day followed in t

## 7. Group chunks into sections — matches your `DocumentSection` schema

In [16]:
# Section type → human-readable title mapping
# Extend this to match whatever section_type values you stored in Pinecone metadata
SECTION_TITLE_MAP = {
    "intro":           "1. Introduction and Rationale",
    "study_design":    "2. Study Design",
    "objectives":      "3. Study Objectives and Endpoints",
    "population":      "4. Study Population",
    "treatment":       "5. Treatment Plan",
    "safety":          "6. Safety and Monitoring",
    "statistics":      "7. Statistical Methodology",
    "ethics":          "8. Ethical Considerations",
    "data_management": "9. Data Management",
    # fallback for chunks that have no section_type metadata
    "general":         "Reference Material",
}


def chunks_to_sections(chunks: list, min_score: float = 0.3) -> list:
    """
    Convert Pinecone chunks into section dicts matching DocumentSection schema.

    Groups chunks by section_type metadata field.
    Chunks below min_score threshold are excluded.

    Returns:
        List of section dicts — each matches DocumentSection in backend/core/models.py
    """
    from collections import defaultdict

    # Filter low-relevance chunks
    good = [c for c in chunks if c['score'] >= min_score]
    print(f"Chunks after score filter (>= {min_score}): {len(good)} / {len(chunks)}")

    # Group by section_type
    groups = defaultdict(list)
    for c in good:
        sec_type = (
            c['metadata'].get('section_type')
            or c['metadata'].get('section')
            or 'general'
        )
        groups[sec_type].append(c)

    sections = []
    for sec_type, sec_chunks in groups.items():
        # Sort chunks by score descending
        sec_chunks.sort(key=lambda x: x['score'], reverse=True)

        # Concatenate chunk texts
        combined_text = "\n\n".join(c['text'] for c in sec_chunks if c['text'].strip())
        word_count    = len(combined_text.split())
        avg_score     = round(sum(c['score'] for c in sec_chunks) / len(sec_chunks), 4)

        # Collect source filenames
        sources = list({
            c['metadata'].get('filename', c['metadata'].get('source', 'Pinecone'))
            for c in sec_chunks
        })

        section = {
            # DocumentSection fields
            "section_id":       sec_type,
            "title":            SECTION_TITLE_MAP.get(sec_type, sec_type.replace('_', ' ').title()),
            "content":          combined_text,
            "section_type":     sec_type,
            "word_count":       word_count,
            "confidence_score": avg_score,
            "sources_used":     sources,
            "compliance_flags": [],
            "revised":          False,
            "revision_count":   0,
            "generated_by":     "pinecone_rag",
            # Extra debug info (strip before sending to frontend if needed)
            "_chunk_count":     len(sec_chunks),
            "_chunk_scores":    [c['score'] for c in sec_chunks],
        }
        sections.append(section)

    # Sort sections by natural document order
    order = list(SECTION_TITLE_MAP.keys())
    sections.sort(
        key=lambda s: order.index(s['section_id']) if s['section_id'] in order else 999
    )
    return sections


# Run it
sections = chunks_to_sections(results, min_score=0.3)

print(f"\n✅ Grouped into {len(sections)} sections:\n")
for s in sections:
    print(f"  [{s['section_id']}]  {s['title']}")
    print(f"     words={s['word_count']}  confidence={s['confidence_score']}  chunks={s['_chunk_count']}")
    print(f"     sources: {s['sources_used']}")
    print()

Chunks after score filter (>= 0.3): 10 / 10

✅ Grouped into 1 sections:

  [general]  Reference Material
     words=2331  confidence=0.8185  chunks=10
     sources: ['Pinecone']



## 8. Print full section content — see exactly what each section contains

In [17]:
for s in sections:
    print("=" * 80)
    print(f"SECTION: {s['title']}")
    print(f"Confidence: {s['confidence_score']}  |  Words: {s['word_count']}  |  Chunks: {s['_chunk_count']}")
    print(f"Sources: {', '.join(s['sources_used'])}")
    print("-" * 80)
    print(s['content'])
    print()

SECTION: Reference Material
Confidence: 0.8185  |  Words: 2331  |  Chunks: 10
Sources: Pinecone
--------------------------------------------------------------------------------
This is a retrospective single arm cohort study collecting clinical, laboratory and physiological data generated in a group of patient affected by cerebrotendinous xanthomatosis, treated with CDCA orally at 750 mg/day followed in the Operative Unit of Neurology and Neuronretabolic Diseases, Medical University of Siena, Policlinico Le ScotteSiena Italy) any anda

(CDCA) in Patients Affected by CerebrotendinousXanthomatosis e VERSION DATE (dd/mm/yy) Approved by the Sponsor SigmaTau and by the CoordinatorPPD to approve the PRTOCOL of the study to conduct tle study in compliance and as agreed with the Sponsor and the Coordinator wn (dd/mm/yy) Signature This Final Version 1.0September 22nd2014 Page 9of44 Osigma-tau ProtocolCDCA-STRCH-CR-14-001 1. Van Bogaert L,Scherer HJ,Epstein E.Une forme cerebrale de la cholesteri

## 9. Metadata-filtered queries — retrieve only specific indication or phase

In [ ]:
# ── Example: only return chunks from Phase III Oncology trials ────────────────
# This uses Pinecone metadata filtering — requires those fields to be in the
# metadata you stored when upserting vectors.
#
# Adjust keys to match YOUR stored metadata field names.

filtered_results = retrieve_from_pinecone(
    query=query_text,
    top_k=10,
    metadata_filter={
        "phase":      {"$eq": "Phase III"},
        "indication": {"$in": ["Oncology", "HER2-positive Metastatic Breast Cancer"]},
    }
)

print(f"Filtered results: {len(filtered_results)} chunks")
for r in filtered_results:
    print(f"  [{r['score']}] {r['id']} — {r['metadata'].get('source', '?')}")
    print(f"         {r['text'][:100]}…")
    print()

## 10. Test multiple indications — batch comparison

In [ ]:
TEST_QUERIES = [
    {
        "label":       "Oncology — HER2+ Breast Cancer Phase III",
        "indication":  "HER2-positive Metastatic Breast Cancer",
        "phase":       "Phase III",
        "design":      "Randomized, Controlled, Double-blind",
        "primary_endpoint":    "Overall Survival",
        "patient_population":  "Adults ≥18, HER2+ mBC, ECOG 0-1",
        "therapeutic_area":    "Oncology",
    },
    {
        "label":       "Cardiovascular — Heart Failure Phase II",
        "indication":  "Heart Failure with Reduced Ejection Fraction",
        "phase":       "Phase II",
        "design":      "Randomized, Open-label",
        "primary_endpoint":    "NYHA Class Improvement",
        "patient_population":  "Adults with HFrEF, EF < 40%",
        "therapeutic_area":    "Cardiovascular",
    },
    {
        "label":       "Diabetes — Type 2 Phase III",
        "indication":  "Type 2 Diabetes Mellitus",
        "phase":       "Phase III",
        "design":      "Randomized, Controlled, Double-blind",
        "primary_endpoint":    "HbA1c reduction",
        "patient_population":  "Adults 18–75, T2DM, HbA1c 7.5–10.5%",
        "therapeutic_area":    "Metabolic",
    },
]

batch_summary = []

for q in TEST_QUERIES:
    label = q.pop('label')
    query = build_query(q)
    chunks = retrieve_from_pinecone(query, top_k=8)
    secs   = chunks_to_sections(chunks, min_score=0.25)

    batch_summary.append({
        'Query':             label,
        'Chunks retrieved':  len(chunks),
        'Sections grouped':  len(secs),
        'Top score':         chunks[0]['score'] if chunks else 0,
        'Avg score':         round(sum(c['score'] for c in chunks) / max(len(chunks), 1), 4),
        'Total words':       sum(s['word_count'] for s in secs),
        'Section IDs':       ', '.join(s['section_id'] for s in secs),
    })

print(pd.DataFrame(batch_summary).to_string(index=False))

## 11. Validate output matches `DocumentSection` schema

In [ ]:
import json

REQUIRED_SECTION_FIELDS = [
    "section_id", "title", "content", "section_type",
    "word_count", "confidence_score", "sources_used",
    "compliance_flags", "revised", "revision_count", "generated_by",
]

all_ok = True
for s in sections:
    missing = [f for f in REQUIRED_SECTION_FIELDS if f not in s]
    if missing:
        print(f"❌ Section '{s['section_id']}' missing fields: {missing}")
        all_ok = False
    else:
        print(f"✅ Section '{s['section_id']}' has all required fields")

if all_ok:
    print("\n✅ All sections validate against DocumentSection schema")
    # Strip debug fields before printing clean output
    clean_sections = [
        {k: v for k, v in s.items() if not k.startswith('_')}
        for s in sections
    ]
    print("\nClean JSON output (first section):")
    print(json.dumps(clean_sections[0], indent=2, default=str))

## 12. Integration snippet — drop this into `rag_service.py`

Once testing confirms retrieval quality, replace or augment the FAISS `retrieve()` method with this.

In [ ]:
INTEGRATION_SNIPPET = '''
# ── Add to backend/services/rag_service.py ────────────────────────────────────
# Install: pip install pinecone-client
# Add PINECONE_API_KEY and PINECONE_INDEX to your .env / config.py

from pinecone import Pinecone as PineconeClient

class PineconeRetriever:
    """
    Drop-in replacement for the FAISS retriever.
    Uses the same SentenceTransformer encoder so embeddings are compatible.
    """

    def __init__(self, api_key: str, index_name: str, encoder, namespace: str = ""):
        self._pc        = PineconeClient(api_key=api_key)
        self._index     = self._pc.Index(index_name)
        self._encoder   = encoder          # reuse the existing SentenceTransformer instance
        self._namespace = namespace

    def retrieve(self, query: str, top_k: int = 5, metadata_filter: dict = None) -> list:
        """
        Returns list of dicts matching existing RAGPipeline.retrieve() output format:
            [{"content": str, "metadata": dict, "score": float}, ...]
        """
        vec = self._encoder.encode([query], normalize_embeddings=True)[0].tolist()
        kwargs = dict(vector=vec, top_k=top_k, include_metadata=True)
        if self._namespace:
            kwargs["namespace"] = self._namespace
        if metadata_filter:
            kwargs["filter"] = metadata_filter

        resp = self._index.query(**kwargs)
        return [
            {
                "content":  m["metadata"].get("text", m["metadata"].get("content", "")),
                "metadata": m["metadata"],
                "score":    m["score"],
            }
            for m in resp.get("matches", [])
        ]


# ── In RAGPipeline.__init__(), add Pinecone as an optional backend ────────────
# After the FAISS init block:

import os
from backend.core.config import cfg

_PINECONE_KEY   = os.getenv("PINECONE_API_KEY", "")
_PINECONE_INDEX = os.getenv("PINECONE_INDEX",   "")

if _PINECONE_KEY and _PINECONE_INDEX:
    self._pinecone = PineconeRetriever(
        api_key    = _PINECONE_KEY,
        index_name = _PINECONE_INDEX,
        encoder    = self._encoder,   # same encoder already loaded
    )
    logger.info("Pinecone retriever initialised — using as primary source")
else:
    self._pinecone = None


# ── In RAGPipeline.retrieve(), add Pinecone results BEFORE FAISS results ──────
def retrieve(self, query: str, top_k: int = 5) -> list:
    results = []

    # 1. Pinecone (historical trials knowledge base) — if configured
    if self._pinecone:
        pc_results = self._pinecone.retrieve(query, top_k=top_k)
        results.extend(pc_results)

    # 2. Local FAISS (user-ingested docs + built-in samples) — always runs
    faiss_results = self._faiss_retrieve(query, top_k=top_k)
    results.extend(faiss_results)

    # 3. Deduplicate by content hash, keep top_k by score
    seen  = set()
    dedup = []
    for r in sorted(results, key=lambda x: x.get("score", 0), reverse=True):
        h = hash(r["content"][:200])
        if h not in seen:
            seen.add(h)
            dedup.append(r)
    return dedup[:top_k]
'''

print(INTEGRATION_SNIPPET)

## 13. (Optional) Upsert test — push a sample document into Pinecone
Run this if you want to test writing to your index before integration.

In [ ]:
# ── Only run this if you want to write test data to your index ────────────────
RUN_UPSERT_TEST = False  # set to True to run

if RUN_UPSERT_TEST:
    SAMPLE_CHUNKS = [
        {
            "id":   "test_chunk_onc_001",
            "text": "Phase III Oncology Protocol: Randomized double-blind study. "
                    "Primary endpoint Overall Survival. N=520 patients. "
                    "HER2+ mBC population. ECOG 0-1.",
            "metadata": {
                "text":         "Phase III Oncology Protocol: Randomized double-blind study. Primary endpoint Overall Survival. N=520 patients. HER2+ mBC population. ECOG 0-1.",
                "section_type": "study_design",
                "indication":   "Oncology",
                "phase":        "Phase III",
                "doc_type":     "Clinical Study Protocol",
                "filename":     "test_phase3_oncology.pdf",
                "year":         "2023",
            }
        },
        {
            "id":   "test_chunk_onc_002",
            "text": "Inclusion: adults ≥18, HER2+ mBC, measurable disease per RECIST v1.1. "
                    "Exclusion: prior PI3K inhibitors, untreated CNS metastases.",
            "metadata": {
                "text":         "Inclusion: adults ≥18, HER2+ mBC, measurable disease per RECIST v1.1. Exclusion: prior PI3K inhibitors, untreated CNS metastases.",
                "section_type": "population",
                "indication":   "Oncology",
                "phase":        "Phase III",
                "doc_type":     "Clinical Study Protocol",
                "filename":     "test_phase3_oncology.pdf",
                "year":         "2023",
            }
        },
    ]

    vectors_to_upsert = []
    for chunk in SAMPLE_CHUNKS:
        vec = embed(chunk['text'])
        vectors_to_upsert.append({
            'id':       chunk['id'],
            'values':   vec,
            'metadata': chunk['metadata'],
        })

    index.upsert(vectors=vectors_to_upsert, namespace=PINECONE_NAMESPACE)
    print(f"✅ Upserted {len(vectors_to_upsert)} test vectors")

    # Verify they are retrievable
    test_results = retrieve_from_pinecone(
        "HER2 breast cancer Phase III randomized protocol",
        top_k=3
    )
    print(f"\nVerification query returned {len(test_results)} results:")
    for r in test_results:
        print(f"  [{r['score']}] {r['id']} — {r['text'][:80]}")
else:
    print("Upsert test skipped (RUN_UPSERT_TEST=False)")

## Summary

| Step | What it does |
|------|--------------|
| 1–3  | Connect to Pinecone, load `all-MiniLM-L6-v2` encoder (same as your FAISS backend) |
| 4    | Build query string from study metadata — identical to `StudyMetadata.to_query_text()` |
| 5–6  | Retrieve and inspect raw chunks with scores |
| 7    | Group chunks by `section_type` → produce `DocumentSection`-shaped dicts |
| 8    | Print full section content for quality review |
| 9    | Filtered queries by indication/phase using Pinecone metadata filters |
| 10   | Batch test across 3 different indications |
| 11   | Schema validation against `DocumentSection` required fields |
| 12   | Drop-in `PineconeRetriever` class + integration patch for `rag_service.py` |
| 13   | Optional upsert test to push sample data into the index |

**Next step:** once retrieval quality looks good in cell 8, copy the `PineconeRetriever` snippet from cell 12 into `backend/services/rag_service.py` and add `PINECONE_API_KEY` + `PINECONE_INDEX` to your `.env`.